<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/ATLASOD.gif?raw=1" style="width:50%"></CENTER>

# Pythia Vistas Event Visualization

How do we turn the predictions of the Standard Model into thousands of simulated particles to study? In this notebook, we will generate proton–proton collisions using the PYTHIA Monte Carlo generator and explore their structure using the Vistas visualization software. 

While ATLAS Open Data offers a myriad of sample types and decay processes, some of those more curious may wish to learn how to generate their own samples to study specific phenomena in varying detail. In this notebook, we'll use a simple PYTHIA example generation with clear configuration categories. Once you've run the example, you can use this script as a test bed to study any events you like, with as much or as little complexity as you desire. Our sample generation produces a Higgs boson through Z boson fusion, followed by the $H\rightarrow b\bar b$ di-jet Higgs decay channel. 

We will learn how to configure the Pythia generator, choose and generate the events we want to see, reconstruct jets, and learn how to interperet our visualization. These events provide a view of the generator's particle history, but simulating the detector response and reconstructing measured objects would require additional steps beyond this notebook. For visualizing reconstructions and flavor tagged tracks, visit our page on using [Phoenix for event visualization](https://opendata.atlas.cern/docs/tutresearch/phoenix).

### Import Vistas and Prebuilt Pythia Installation

The next cell installs a prebuilt PYTHIA package and downloads Vistas. The imported package is called `pythia8mc`, but we will use the shorter `pythia8` throughout the notebook. Currently, there is no prebuilt version of Pythia for linux or windows for modern Python, so the following code uses a workaround by installing an older version of Pythia and importing Vistas.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import sys

vistas_dir = Path.cwd()
urlretrieve(
    "https://gitlab.com/Pythia8/releases/-/raw/pythia8318/plugins/python/extra/Vistas.py",
    vistas_dir / "Vistas.py",
)
sys.path.insert(0, str(vistas_dir))

from Vistas import Vistas

%pip install --only-binary=:all: pythia8mc

import pythia8mc as pythia8

## Sample Script

Below is a sample code representing a $\sqrt{s}=7$ TeV center-of-mass energy proton-proton beam, and we specify this using `Beams:eCM = 7000`. We focus the process on Higgs boson production through Z boson fusion using the `HiggsSM:ff2Hff(t:ZZ) = on` readString setting. We choose the decay after this, using the onMode and onIfMatch settings. `25:onMode = off` disables all of the Higgs decay channels, and `25:onIfMatch = 5 -5` enables the $H\rightarrow b\bar b$ decay mode. These numbers are particle identifiers: `2212` denotes a proton, `25` the Higgs boson, and `5` and `-5` a bottom quark and antibottom quark. The quarks will then shower and hadronize, producing particles that can form jets.

In PYTHIA, particle flavors are represented using integer ids, while processes have distinct names. You can find availaible [hard scattering processes](https://pythia8.web.cern.ch/manuals/pythia8318/ProcessSelection.html) and [particle ids](https://pythia8.web.cern.ch/manuals/pythia8318/ParticleData.html) at the sites linked. For PYTHIA to generate an event, the events have to represent valid physical processes, and obey established [physical laws](https://cds.cern.ch/record/2759491/files/Conservation%20Laws%20-%20ATLAS%20Physics%20Cheat%20Sheet.pdf). If we set the `25:onIfMatch` flag to select for top quark decays, which have a higher mass than the Higgs boson, we'll get nothing but an empty simulator!

After this sample event, we'll give a rundown on reading Vistas visualizations, and some tips for using Pythia. For a comprehensive manual on configuring Pythia, refer to the [Pythia 8 Index](https://pythia8.web.cern.ch/manuals/pythia8318/Welcome.html). Feel free to use the following code as a template for your own use! Configuring Pythia can be complex, and load times can vary significantly, so don't get discouraged if your code doesn't run quickly the first time!

In [ ]:
from time import perf_counter

# --------------------------------- Pythia Importer: Don't Remove! ----------------------------------------
pythia = pythia8.Pythia()

# --------------------------------- Seed Randomizer: Ensures each event is unique ----------------------------------------
import secrets
seed = secrets.randbelow(900_000_000) + 1
pythia.readString("Random:setSeed = on")
pythia.readString(f"Random:seed = {seed}")
print("Seed:", seed)  # Keep this if you may want to reproduce the run

# --------------------------------- Event Generation Settings ----------------------------------------
pythia.readString("Print:quiet = on") # Change to "off" for comprehensive particle-by-particle output.
pythia.readString("Tune:pp = 21") # Sets the current GenTune and PDF to the ATLAS A14 tune with NNPDF2.3LO. Refer to "Setup Run Tasks" in the index for a list of available tunes.

# --------------------------------- Beam Settings ---------------------------------------- 
pythia.readString("Beams:idA = 2212") # Beam 1 Particles
pythia.readString("Beams:idB = 2212") # Beam 2 Particles
pythia.readString("Beams:eCM = 7000") # Beam Center-of-Mass Energy

# --------------------------------- Decay Mode Settings ----------------------------------------
pythia.readString("HiggsSM:ff2Hff(t:ZZ) = on")  # Primary phenomena/event type

pythia.readString("25:onMode = off") # Removes all decay modes of the particle type indicated
pythia.readString("25:onIfMatch = 5 -5") # Starts with the particle type, and turns on the decay mode indicated after the "=" if it can occur

pythia.readString("PartonLevel:MPI = on") # Lessens soft activity and removes multi-particle interactions when off. Set to "on" to increase accuracy of the simulation.
# WARNING: Setting this to "on" may increase load times significantly.

# Indicates how many unique events you want to generate when this cell is run.
n_events = 3

# --------------------------------- Running Settings: Change if you know what you're doing ----------------------------------------

# Indicates if Pythia is functioning 
print("Initializing...", flush=True)
start = perf_counter()
if not pythia.init():
    raise RuntimeError("PYTHIA initialization failed")
print(f"Initialization: {perf_counter() - start:.1f} s", flush=True)

# Generates events and returns data about the generation
generated_events = []
for i in range(n_events):
    start = perf_counter()
    if not pythia.next():
        raise RuntimeError(f"Event {i + 1} failed")
    generated_events.append(pythia8.Event(pythia.event))  # Save this event
    print(f"Event {i + 1}: {perf_counter() - start:.1f} s", flush=True)

## Vistas Visualization

The code below handles the visualization of the events we generated above. The pre-written code includes a basic configuration of the jet visualizer, and an example setting of the color filter, two of many configurable parts of the vistas software. For a full list of applicable settings, refer to [the official documentation.](https://arxiv.org/pdf/2606.19524#page=13&zoom=page-fit) After the following cell runs, you should get a URL, along with a readout of the amount of particles that make up each event. 

*Note: Vistas is still in active development, so some errors may occur! If you get an error readout, there's a good chance it may not directly effect your visualization. In either case, you can always [report bugs to the PYTHIA development team.](https://pythia.org/contact.html)*

In [ ]:
# --------------------------------- Visualization Settings ----------------------------------------
viewer = Vistas(pythia)
viewer.opts["jets"].update({
    "algorithm": "akt",   # Determines what algorithm is used to draw jets (none, akt, ca, kt)
    "r": 0.4, # Jet Radius scale value
    "ptmin": 30.0, # Minimum Transverse Momentum of drawn jets
    "etamax": 5.0, # Maximum Pseudorapidity of drawn jets
    "select": "visible", # Determines which particle types to determine jets based on (all, visible, charged)
    "mass": "gen", # Determines what mass to assign to jet particles (zero, pion, gen)
})

viewer.opts["color"].update({
    "scale": "constant",   # Determines how to scale the track colors (log, fixed, constant)
    "observable": "p.pT()", # Determines what aspect of the track to attach the color values to. Energy (e) and transverse momentum (pT) are common.
})

# --------------------------------- Event JSON Compiler ----------------------------------------
events = {}
for i, saved_event in enumerate(generated_events, start=1):
    data = viewer.data(event=saved_event)  # Visualize the saved event
    if not data:
        continue

    event = data["Pythia 8 Event"]
    event["event number"] = i
    events[f"Event {i}"] = event
    print(f"Event {i}: {saved_event.size()} particles, "
          f"{len(event['Jets']['boosted frame jets'])} jet cones")

if not events:
    raise RuntimeError("No events could be visualized")

# Creates the temporary visualizer file, and prints a link to the Phoenix viewer.
# 'sleep=60' indicates that the link expires after 60 seconds! Change if you want this delayed. 
# Note: You will not be able to run the code again during the sleep interval without restarting.
viewer.display(data=events, pad=None, sleep=60)

# Uncomment the following line and give it a name to save your visualization file!
# viewer.write("name", data=events)

## Interpereting Vistas Visualizations

In Vistas, events are visualized based on the categories of processes [inside a collision](https://cds.cern.ch/record/2951043/files/Inside%20a%20Collision.pdf). Each part of an event is identified by colored line or "track" , and will be discussed below, along with a short description of what each step is. Alternatively, each group of tracks can be toggled on and off via the Phoenix Menu, allowing you to visualize specific steps.

By default, Vistas colors particle tracks logarithmically based on their energy level. The observable this color change is based on, and way that this change effects colors can be configured using the [`color`](https://arxiv.org/pdf/2606.19524#page=13&zoom=page-fit) option in your visualization code. 

- <span style="color:#bf33c0">**Hard Process:**</span> The highest energy, often most studied part of the collision. This is where you can expect to see massive particle production, like Higgs or vector boson production. They appear a pink/purple hue in the visualization.
- <span style="color:#2200af">**Beams:**</span> Reperesnts the initial trajectories of the configured beams. Will appear based on your choice of Parton Distribution Function, appearing blue in the visualization.
- <span style="color:red">**Multi Parton Interactions:**</span> Following the hard process, the constituent partons of the initial collision will collide with each other, creating a range of non-prompt soft activity. Tracks from these interactions will appear red in the visualization software. *Note: These will not appear if MPI is set to `off` in the generation code*
- <span style="color:orange">**Initial State Radiation (ISR):**</span> Before the hard process, as particles move towards each other, some partons may emit additional particles earlier. For example, while quarks fly through the detector, they often emit gluons due to their interactions with the strong force, even before colliding with an incoming proton. These initial radiations are colored orange in the visualization.
- <span style="color:#a3b62a">**Final State Radiation (FSR):**</span> After the hard process, the products present in the final state of the process will often decay into a variety of other particles, comprising the "final state radiation". Final state radiation is represented in yellow in the visualization.
- <span style="color:green">**Hadronization:**</span> After the hard process occurs, and the initial observable event takes place, color-charged particles begin to recombine into composite particles. The creation and spread of composite hadrons from quarks and gluons is known as hadronization, and is represented in green in the visualization.
- <span style="color:cyan">**Decay:**</span> Many hadrons produced during hadronization are not stable, and will undergo decays to reach more stable states. These decays are represented by bluish-green tracks in the visualization.

### How Jets are drawn

A jet algorithm groups particles into objects that summarize concentrated flows of momentum. In the sample code we use anti-$k_T$ with radius parameter $R=0.4$ and retain jets with $p_T>30$ GeV, using visible final-state particles.

The cones show the reconstructed jet directions and chosen size parameter, though not all particles appearing in a jet cone will be created by the jet. The default hard process we are looking at contains two outgoing quarks in addition to the $b\bar b$ pair from the Higgs decay. However, this does not immediately tell us how many jets we will get, as radiation, clustering, and the selection requirements affect the number displayed. This algorithm also does not tell us what kind of jet (b-jet, c-jet, etc) each jet is, so we have to rely on deeper analyses for this!

## Pythia/Vistas Tips and Tricks

- Always re-run the generation cell after making edits, before the visualization cell. Changing generation settings only edits the instructions for generation; your most recent run of the generator will still be saved in the notebook instance until you run the cell again. Of course, changing visualization settings only requires you to rerun the visualization cell.
- If you find a pattern or result you especially like, take note of the seed and configuration settings to recreate it the next time you want to study it! The supplied sample code randomizes the seed for each run, so you'll probably never see the same seed twice. 
- Using the `onMode = off` setting allows you to select for specific decay channels instead of sorting through events for your prompt decay! Of course, if you want a bigger picture showing all of the ways a parton can decay, it'll be better to leave this setting `on`.
- Open the Phoenix link while the visualization cell is still running, and the kernel says "busy". With `sleep=60`, Vistas waits 60 seconds before deleting its temporary uploads. Once the data have loaded in the Phoenix tab, your visualization will persist in the opened tab until you close the tab. Wait for the cell to finish running or restart the kernel before running another cell.